# PySpark DataFrame DSL — Part 3
# Expressions & withColumn

## Topics Covered

1. `expr()` — Build expressions using SQL-style syntax
2. `selectExpr()` — Select columns and expressions using SQL-style syntax
3. `withColumn()` — Create or modify columns
4. `when()` — Build conditional expressions
5. `otherwise()` — Define the alternative result of a condition

Each topic contains a detailed explanation and 5 different examples.

---

# Example DataFrame

```python
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("DataFrameDSLPart3") \
    .getOrCreate()

data = [
    ("C001", "Karthik", "Chennai", 25, 55000.50, 5),
    ("C002", "Arun", "Mumbai", 28, 62000.75, 8),
    ("C003", "Rahul", "Delhi", 26, 48000.25, 3),
    ("C004", "Priya", "Bangalore", 30, 75000.00, 10),
    ("C005", "Anitha", "Chennai", 24, 45000.00, 2),
    ("C006", "Vijay", "Mumbai", 32, 82000.00, 12)
]

columns = ["customer_id", "name", "city", "age", "salary", "experience"]

df = spark.createDataFrame(data, columns)
df.show()
```

---


# 1. `expr()` — SQL-Style Column Expressions

## Definition

`expr()` converts a SQL-style expression written as a string into a Spark Column expression.

```python
from pyspark.sql.functions import expr
```

Syntax:

```python
expr("SQL expression")
```

It is useful when SQL syntax is more natural than building the expression with `col()`.

### Example 1 — Arithmetic

```python
df.select(
    "name",
    expr("salary * 12").alias("annual_salary")
).show()
```

### Example 2 — Add Bonus

```python
df.select(
    "name",
    expr("salary + 10000").alias("salary_with_bonus")
).show()
```

### Example 3 — Multiple Columns

```python
df.select(
    "name",
    expr("age + experience").alias("age_plus_experience")
).show()
```

### Example 4 — CASE Expression

```python
df.select(
    "name",
    expr("""
        CASE
            WHEN salary >= 70000 THEN 'High'
            WHEN salary >= 50000 THEN 'Medium'
            ELSE 'Low'
        END
    """).alias("salary_category")
).show()
```

### Example 5 — String Functions

```python
df.select(
    "name",
    expr("upper(name)").alias("upper_name"),
    expr("length(name)").alias("name_length")
).show()
```

### Mental Model

```text
SQL-style string
       ↓
     expr()
       ↓
Spark Column Expression
       ↓
DataFrame operation
```

### Important Points

- `expr()` accepts a string.
- The string uses Spark SQL expression syntax.
- It can be used with `select()`.
- It can be used with `withColumn()`.
- It is useful for SQL-style calculations and expressions.

---

# 2. `selectExpr()` — Select Columns and Expressions

## Definition

`selectExpr()` lets you select columns and write SQL-style expressions directly as strings.

Syntax:

```python
df.selectExpr("column1", "column2", "expression")
```

It is especially useful when you want `SELECT`-style expressions without separately calling `expr()`.

### Example 1 — Select Columns

```python
df.selectExpr(
    "name",
    "city"
).show()
```

### Example 2 — Calculated Column

```python
df.selectExpr(
    "name",
    "salary * 12 AS annual_salary"
).show()
```

### Example 3 — Rename Columns

```python
df.selectExpr(
    "customer_id AS id",
    "name AS customer_name",
    "city"
).show()
```

### Example 4 — Multiple Expressions

```python
df.selectExpr(
    "name",
    "age + 1 AS next_year_age",
    "salary * 12 AS annual_salary",
    "experience + 1 AS next_year_experience"
).show()
```

### Example 5 — CASE Expression

```python
df.selectExpr(
    "name",
    """
    CASE
        WHEN experience >= 10 THEN 'Senior'
        WHEN experience >= 5 THEN 'Mid-Level'
        ELSE 'Junior'
    END AS experience_level
"""
).show()
```

### Comparison

Using `select()` + `expr()`:

```python
df.select(
    "name",
    expr("salary * 12").alias("annual_salary")
)
```

Using `selectExpr()`:

```python
df.selectExpr(
    "name",
    "salary * 12 AS annual_salary"
)
```

### Mental Model

```text
select()
   +
expr()
   ↓
selectExpr()
```

### Important Points

- Expressions are supplied as strings.
- SQL-style syntax is used.
- `AS` can define aliases.
- It returns a new DataFrame.
- Columns not selected are not included in the result.

---

# 3. `withColumn()` — Create or Modify Columns

## Definition

`withColumn()` creates a new column or replaces an existing column with a new expression.

Syntax:

```python
df.withColumn("column_name", expression)
```

It returns a new DataFrame.

### Example 1 — Create a New Column

```python
from pyspark.sql.functions import col

df2 = df.withColumn(
    "annual_salary",
    col("salary") * 12
)

df2.show()
```

### Example 2 — Use `expr()`

```python
df2 = df.withColumn(
    "annual_salary",
    expr("salary * 12")
)

df2.show()
```

### Example 3 — Modify an Existing Column

```python
df2 = df.withColumn(
    "salary",
    col("salary") + 5000
)

df2.show()
```

Because the target is `"salary"`, the existing salary column is replaced.

### Example 4 — Constant Column

```python
from pyspark.sql.functions import lit

df2 = df.withColumn(
    "country",
    lit("India")
)

df2.show()
```

### Example 5 — Expression Using Multiple Columns

```python
df2 = df.withColumn(
    "salary_per_experience",
    col("salary") / col("experience")
)

df2.show()
```

### Important Points

- Creates a column.
- Can replace an existing column.
- Returns a new DataFrame.
- Can accept `col()`, `expr()`, `when()`, `lit()`, and other expressions.
- Does not modify the original DataFrame in place.

---

# 4. `when()` — Conditional Expressions

## Definition

`when()` creates a conditional Column expression.

Conceptually:

```text
IF condition is TRUE
THEN return a value
```

It is similar to SQL `CASE WHEN`.

```python
from pyspark.sql.functions import when
```

Syntax:

```python
when(condition, value)
```

It is commonly combined with `otherwise()`.

### Example 1 — Age Category

```python
df2 = df.withColumn(
    "age_category",
    when(
        col("age") >= 30,
        "Senior"
    ).otherwise(
        "Junior"
    )
)

df2.show()
```

### Example 2 — Salary Category

```python
df2 = df.withColumn(
    "salary_category",
    when(
        col("salary") >= 70000,
        "High"
    ).otherwise(
        "Normal"
    )
)

df2.show()
```

### Example 3 — Multiple Conditions

```python
df2 = df.withColumn(
    "salary_category",
    when(col("salary") >= 70000, "High")
    .when(col("salary") >= 50000, "Medium")
    .otherwise("Low")
)

df2.show()
```

Conditions are evaluated in order.

### Example 4 — Experience Level

```python
df2 = df.withColumn(
    "experience_level",
    when(col("experience") >= 10, "Expert")
    .when(col("experience") >= 5, "Experienced")
    .otherwise("Fresher")
)

df2.show()
```

### Example 5 — City-Based Category

```python
df2 = df.withColumn(
    "location_group",
    when(col("city") == "Chennai", "South")
    .when(col("city") == "Bangalore", "South")
    .otherwise("Other")
)

df2.show()
```

### Important Points

- `when()` creates conditional logic.
- It can be used with `select()`.
- It can be used with `withColumn()`.
- Multiple `when()` expressions can be chained.
- Conditions are evaluated from top to bottom.
- `otherwise()` defines the fallback.

---

# 5. `otherwise()` — Alternative Result

## Definition

`otherwise()` defines the value returned when none of the preceding `when()` conditions are TRUE.

Basic pattern:

```python
when(condition, value).otherwise(fallback)
```

### Example 1 — Simple Fallback

```python
df2 = df.withColumn(
    "status",
    when(col("age") >= 30, "Senior")
    .otherwise("Junior")
)

df2.show()
```

### Example 2 — Salary Classification

```python
df2 = df.withColumn(
    "salary_category",
    when(col("salary") >= 70000, "High")
    .otherwise("Normal")
)

df2.show()
```

### Example 3 — Multiple `when()` + `otherwise()`

```python
df2 = df.withColumn(
    "salary_category",
    when(col("salary") >= 70000, "High")
    .when(col("salary") >= 50000, "Medium")
    .otherwise("Low")
)

df2.show()
```

### Example 4 — Experience Classification

```python
df2 = df.withColumn(
    "experience_level",
    when(col("experience") >= 10, "Expert")
    .when(col("experience") >= 5, "Experienced")
    .otherwise("Junior")
)

df2.show()
```

### Example 5 — Numeric Fallback

`otherwise()` can return a numeric value.

```python
df2 = df.withColumn(
    "bonus",
    when(col("salary") >= 70000, 10000)
    .otherwise(5000)
)

df2.show()
```

---

# `when()` Without `otherwise()`

`otherwise()` is optional syntactically.

```python
df2 = df.withColumn(
    "category",
    when(col("salary") >= 70000, "High")
)
```

Rows where the condition is FALSE receive `NULL`.

Conceptually:

```text
salary >= 70000
       |
       +---- TRUE  → High
       |
       +---- FALSE → NULL
```

Use `otherwise()` when you need an explicit fallback value.

---

# 6. Combining the Expressions

## `expr()` + `withColumn()`

```python
df2 = df.withColumn(
    "annual_salary",
    expr("salary * 12")
)

df2.show()
```

Mental model:

```text
withColumn()
      +
expr()
      ↓
New calculated column
```

## `selectExpr()` + `withColumn()`

```python
df2 = df \
    .selectExpr(
        "customer_id",
        "name",
        "salary",
        "salary * 12 AS annual_salary"
    ) \
    .withColumn(
        "salary_category",
        when(col("salary") >= 70000, "High")
        .otherwise("Normal")
    )

df2.show()
```

Pipeline:

```text
Original DataFrame
       ↓
selectExpr()
       ↓
Create annual_salary
       ↓
withColumn()
       ↓
Create salary_category
       ↓
Final DataFrame
```

## `when()` + `select()`

```python
df.select(
    "name",
    "salary",
    when(col("salary") >= 70000, "High")
    .otherwise("Normal")
    .alias("salary_category")
).show()
```

## All Major Expressions Together

```python
from pyspark.sql.functions import col, expr, when, lit

df2 = df \
    .withColumn(
        "annual_salary",
        expr("salary * 12")
    ) \
    .withColumn(
        "salary_category",
        when(col("salary") >= 70000, "High")
        .when(col("salary") >= 50000, "Medium")
        .otherwise("Low")
    ) \
    .withColumn(
        "country",
        lit("India")
    )

df2.show()
```

This combines:

```text
expr()       → calculated expression
when()       → condition
otherwise()  → fallback
lit()        → constant
withColumn() → add/modify column
```

---

# 7. `expr()` vs `col()`

Both can create column expressions.

### `col()`

```python
df.select(
    (col("salary") * 12).alias("annual_salary")
)
```

### `expr()`

```python
df.select(
    expr("salary * 12").alias("annual_salary")
)
```

Mental model:

```text
col()
 ↓
Python-style column expression

expr()
 ↓
SQL-style expression
```

---

# 8. `selectExpr()` vs `select()` + `expr()`

### `select()` + `expr()`

```python
df.select(
    "name",
    expr("salary * 12").alias("annual_salary")
)
```

### `selectExpr()`

```python
df.selectExpr(
    "name",
    "salary * 12 AS annual_salary"
)
```

Both express the same transformation.

A useful rule:

```text
Python column-expression style
    ↓
select() + col()/expr()

SQL-style expression syntax
    ↓
selectExpr()
```

---

# 9. `withColumn()` vs `selectExpr()`

## `withColumn()`

Use it when you want to add or replace a column while retaining the other columns.

```python
df2 = df.withColumn(
    "annual_salary",
    col("salary") * 12
)
```

Conceptually:

```text
customer_id
name
city
age
salary
experience
annual_salary
```

## `selectExpr()`

Use it when you want to explicitly define the columns and expressions in the result.

```python
df2 = df.selectExpr(
    "customer_id",
    "name",
    "salary * 12 AS annual_salary"
)
```

Result:

```text
customer_id
name
annual_salary
```

---

# 10. Practical Data Engineering Example

Requirement:

1. Calculate annual salary.
2. Categorize salary.
3. Categorize experience.
4. Add country.
5. Keep only required columns.

```python
df2 = df \
    .withColumn(
        "annual_salary",
        col("salary") * 12
    ) \
    .withColumn(
        "salary_category",
        when(col("salary") >= 70000, "High")
        .when(col("salary") >= 50000, "Medium")
        .otherwise("Low")
    ) \
    .withColumn(
        "experience_level",
        when(col("experience") >= 10, "Expert")
        .when(col("experience") >= 5, "Experienced")
        .otherwise("Junior")
    ) \
    .withColumn(
        "country",
        lit("India")
    ) \
    .select(
        "customer_id",
        "name",
        "city",
        "salary",
        "annual_salary",
        "salary_category",
        "experience_level",
        "country"
    )

df2.show()
```

This is a realistic DataFrame DSL transformation pipeline.

---

# 11. SQL Equivalent of Conditional Logic

PySpark:

```python
df.withColumn(
    "salary_category",
    when(col("salary") >= 70000, "High")
    .when(col("salary") >= 50000, "Medium")
    .otherwise("Low")
)
```

SQL:

```sql
SELECT
    *,
    CASE
        WHEN salary >= 70000 THEN 'High'
        WHEN salary >= 50000 THEN 'Medium'
        ELSE 'Low'
    END AS salary_category
FROM customers;
```

Mental mapping:

```text
PySpark              SQL
--------------------------------
when()           →    CASE WHEN
otherwise()      →    ELSE
alias()          →    AS
col()            →    column reference
expr()           →    SQL expression
selectExpr()     →    SELECT expressions
```

---

# 12. Important Rules

## Rule 1 — `withColumn()` returns a new DataFrame

```python
df2 = df.withColumn(
    "annual_salary",
    col("salary") * 12
)
```

The original `df` is not modified in place.

## Rule 2 — Use `otherwise()` when a fallback is required

```python
when(
    condition,
    value
).otherwise(
    fallback
)
```

## Rule 3 — Multiple `when()` conditions are evaluated in order

```python
when(condition1, value1) \
    .when(condition2, value2) \
    .otherwise(value3)
```

The first matching condition determines the result.

## Rule 4 — `selectExpr()` uses SQL-style strings

```python
df.selectExpr(
    "salary * 12 AS annual_salary"
)
```

## Rule 5 — `expr()` converts a SQL-style expression into a Column expression

```python
expr("salary * 12")
```

---

# 13. Quick Reference

| Operation | Purpose |
|---|---|
| `expr()` | Convert a SQL-style expression into a Column expression |
| `selectExpr()` | Select columns and SQL-style expressions |
| `withColumn()` | Create or replace a DataFrame column |
| `when()` | Define a conditional expression |
| `otherwise()` | Define the fallback result |

---

# 14. Method Comparison

| Requirement | Recommended Operation |
|---|---|
| Select existing columns | `select()` |
| Select using SQL-style expressions | `selectExpr()` |
| Create SQL-style expression | `expr()` |
| Add a new column | `withColumn()` |
| Replace an existing column | `withColumn()` |
| Conditional logic | `when()` |
| Fallback condition | `otherwise()` |

---

# 15. Final Mental Model

```text
                         DataFrame
                             |
                 +-----------+-----------+
                 |                       |
                 ↓                       ↓
              select()             withColumn()
                 |                       |
                 |                 +-----+-----+
                 |                 |           |
                 ↓                 ↓           ↓
              expr()             when()      col()
                 |                 |
                 ↓                 ↓
        SQL-style expression   condition
                 |                 |
                 +--------+--------+
                          |
                          ↓
                     otherwise()
                          |
                          ↓
                  Final Expression
                          |
                          ↓
                    DataFrame
```

---

# Key Takeaways

1. `expr()` lets you write Spark SQL-style expressions inside the DataFrame API.
2. `selectExpr()` lets you select columns and SQL-style expressions directly as strings.
3. `withColumn()` creates a new column or replaces an existing column.
4. `when()` is used for conditional logic.
5. `otherwise()` provides the fallback value when no `when()` condition matches.
6. Multiple `when()` expressions can be chained.
7. `expr()` and `col()` can often express the same calculation using different syntax.
8. `selectExpr()` is convenient when you prefer SQL-style expressions.
9. `withColumn()` is useful when adding or modifying columns while retaining the rest of the DataFrame.
10. These operations can be chained together to build real ETL transformation pipelines.

## Core Pattern

```python
from pyspark.sql.functions import col, expr, when, lit

result = df \
    .withColumn(
        "annual_salary",
        expr("salary * 12")
    ) \
    .withColumn(
        "salary_category",
        when(col("salary") >= 70000, "High")
        .when(col("salary") >= 50000, "Medium")
        .otherwise("Low")
    ) \
    .withColumn(
        "country",
        lit("India")
    )

result.show()
```

This is a fundamental pattern in PySpark DataFrame DSL for building transformation pipelines.
